In [3]:
!pip install requests beautifulsoup4 pandas

In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import os
import re

In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL = "https://books.toscrape.com/"

# 3 categories required by the assignment
categories = {
    "Travel": "travel_2/index.html",
    "Mystery": "mystery_3/index.html",
    "Historical Fiction": "historical-fiction_4/index.html"
}

books = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

for category_name, category_path in categories.items():

    url = BASE_URL + "catalogue/category/books/" + category_path

    while url:

        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        for article in soup.select("article.product_pod"):

            title = article.h3.a["title"]

            price = article.select_one(
                ".price_color"
            ).get_text(strip=True)

            rating = article.select_one(
                "p.star-rating"
            )["class"][1]

            availability = article.select_one(
                ".availability"
            ).get_text(" ", strip=True)

            books.append({
                "title": title,
                "price": price,
                "star_rating": rating,
                "availability": availability,
                "category": category_name
            })

        # Move to next page
        next_button = soup.select_one("li.next a")

        if next_button:
            next_page = next_button["href"]

            url = url.rsplit("/", 1)[0] + "/" + next_page
        else:
            url = None


# Create DataFrame
df = pd.DataFrame(books)

print("Total books scraped:", len(df))
print("Total categories:", df["category"].nunique())

display(df.head(10))

Total books scraped: 69
Total categories: 3


,title,price,star_rating,availability,category
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel
5,A Summer In Europe,Â£44.34,Two,In stock,Travel
6,The Great Railway Bazaar,Â£30.54,One,In stock,Travel
7,A Year in Provence (Provence #1),Â£56.88,Four,In stock,Travel
8,The Road to Little Dribbling: Adventures of an...,Â£23.21,One,In stock,Travel
9,Neither Here nor There: Travels in Europe,Â£38.95,Three,In stock,Travel


In [9]:
# Step 4: Clean the scraped data

# 1. Clean price and convert it to float
def clean_price(value):
    value = str(value)

    # Remove currency symbols and unwanted encoding characters
    value = value.replace("£", "")
    value = value.replace("Â", "")
    value = value.replace("â‚¬", "")
    value = value.strip()

    return float(value)


df["price_gbp"] = df["price"].apply(clean_price)


# 2. Convert text rating to integer
rating_mapping = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["star_rating"].map(rating_mapping)


# 3. Convert availability to Boolean
df["in_stock"] = df["availability"].str.contains(
    "In stock",
    case=False,
    na=False
)


# 4. Display cleaned data
print("Cleaned data:")
display(
    df[
        [
            "title",
            "price_gbp",
            "rating",
            "in_stock",
            "category"
        ]
    ].head(10)
)


# 5. Check data types
print("\nData types:")
print(
    df[
        ["price_gbp", "rating", "in_stock"]
    ].dtypes
)


# 6. Check missing values
print("\nMissing values:")
print(
    df[
        ["price_gbp", "rating", "in_stock"]
    ].isna().sum()
)

Cleaned data:


,title,price_gbp,rating,in_stock,category
0,It's Only the Himalayas,45.17,2,True,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,4,True,Travel
2,See America: A Celebration of Our National Par...,48.87,3,True,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,2,True,Travel
4,Under the Tuscan Sun,37.33,3,True,Travel
5,A Summer In Europe,44.34,2,True,Travel
6,The Great Railway Bazaar,30.54,1,True,Travel
7,A Year in Provence (Provence #1),56.88,4,True,Travel
8,The Road to Little Dribbling: Adventures of an...,23.21,1,True,Travel
9,Neither Here nor There: Travels in Europe,38.95,3,True,Travel



Data types:
price_gbp    float64
rating         int64
in_stock        bool
dtype: object

Missing values:
price_gbp    0
rating       0
in_stock     0
dtype: int64


In [10]:
# Step 5: Convert GBP to INR using the required fixed rate

GBP_TO_INR = 105.50

df["price_inr"] = df["price_gbp"] * GBP_TO_INR

# Display the converted prices
display(
    df[
        [
            "title",
            "price_gbp",
            "price_inr"
        ]
    ].head(10)
)

# Verify the conversion
print("Conversion rate used: 1 GBP = 105.50 INR")
print("price_inr data type:", df["price_inr"].dtype)

,title,price_gbp,price_inr
0,It's Only the Himalayas,45.17,4765.435
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.865
2,See America: A Celebration of Our National Par...,48.87,5155.785
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.170
4,Under the Tuscan Sun,37.33,3938.315
5,A Summer In Europe,44.34,4677.870
6,The Great Railway Bazaar,30.54,3221.970
7,A Year in Provence (Provence #1),56.88,6000.840
8,The Road to Little Dribbling: Adventures of an...,23.21,2448.655
9,Neither Here nor There: Travels in Europe,38.95,4109.225


Conversion rate used: 1 GBP = 105.50 INR
price_inr data type: float64


In [11]:
# Step 6: Create normalized SQLite database

import sqlite3

# Create SQLite database
conn = sqlite3.connect("books_database.sqlite")

# Enable foreign key constraints
conn.execute("PRAGMA foreign_keys = ON")

cursor = conn.cursor()

# Drop tables if they already exist
cursor.execute("DROP TABLE IF EXISTS books")
cursor.execute("DROP TABLE IF EXISTS categories")

# Create categories table
cursor.execute("""
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE NOT NULL
)
""")

# Create books table
cursor.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
)
""")

# Insert unique categories
unique_categories = df["category"].drop_duplicates().tolist()

for category_id, category_name in enumerate(unique_categories, start=1):
    cursor.execute(
        """
        INSERT INTO categories (category_id, category_name)
        VALUES (?, ?)
        """,
        (category_id, category_name)
    )

# Create category → ID mapping
category_mapping = {
    category_name: category_id
    for category_id, category_name
    in enumerate(unique_categories, start=1)
}

# Insert books
for book_id, (_, row) in enumerate(df.iterrows(), start=1):

    cursor.execute(
        """
        INSERT INTO books
        (
            book_id,
            title,
            price_gbp,
            price_inr,
            rating,
            in_stock,
            category_id
        )
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """,
        (
            book_id,
            row["title"],
            row["price_gbp"],
            row["price_inr"],
            row["rating"],
            int(row["in_stock"]),
            category_mapping[row["category"]]
        )
    )

# Save changes
conn.commit()

print("SQLite database created successfully!")
print("Database file: books_database.sqlite")

# Check number of rows
print("\nCategories:", cursor.execute(
    "SELECT COUNT(*) FROM categories"
).fetchone()[0])

print("Books:", cursor.execute(
    "SELECT COUNT(*) FROM books"
).fetchone()[0])

SQLite database created successfully!
Database file: books_database.sqlite

Categories: 3
Books: 69


In [12]:
# Step 7: Verify SQLite tables and relationships

print("TABLES IN DATABASE:")

tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""", conn)

display(tables)


print("\nCATEGORIES TABLE:")

categories_check = pd.read_sql_query("""
    SELECT *
    FROM categories
""", conn)

display(categories_check)


print("\nBOOKS TABLE (first 5 rows):")

books_check = pd.read_sql_query("""
    SELECT *
    FROM books
    LIMIT 5
""", conn)

display(books_check)


print("\nFOREIGN KEY RELATIONSHIP:")

foreign_keys = pd.read_sql_query("""
    PRAGMA foreign_key_list(books)
""", conn)

display(foreign_keys)

TABLES IN DATABASE:


,name
0,books
1,categories



CATEGORIES TABLE:


,category_id,category_name
0,1,Travel
1,2,Mystery
2,3,Historical Fiction



BOOKS TABLE (first 5 rows):


,book_id,title,price_gbp,price_inr,rating,in_stock,category_id
0,1,It's Only the Himalayas,45.17,4765.435,2,1,1
1,2,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.865,4,1,1
2,3,See America: A Celebration of Our National Par...,48.87,5155.785,3,1,1
3,4,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.170,2,1,1
4,5,Under the Tuscan Sun,37.33,3938.315,3,1,1



FOREIGN KEY RELATIONSHIP:


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,categories,category_id,category_id,NO ACTION,NO ACTION,NONE


In [13]:
# SQL Query 1: SELECT + WHERE + ORDER BY + LIMIT
# Find the 10 highest-rated books that are in stock

query1 = """
SELECT
    book_id,
    title,
    price_gbp,
    price_inr,
    rating,
    in_stock
FROM books
WHERE in_stock = 1
ORDER BY rating DESC, price_inr DESC
LIMIT 10
"""

result1 = pd.read_sql_query(query1, conn)

print("SQL Query 1:")
print(query1)

print("\nQuery 1 Output:")
display(result1)

SQL Query 1:

SELECT
    book_id,
    title,
    price_gbp,
    price_inr,
    rating,
    in_stock
FROM books
WHERE in_stock = 1
ORDER BY rating DESC, price_inr DESC
LIMIT 10


Query 1 Output:


,book_id,title,price_gbp,price_inr,rating,in_stock
0,46,A Flight of Arrows (The Pathfinders #2),55.53,5858.415,5,1
1,30,The Bachelor Girl's Guide to Murder (Herringfo...,52.30,5517.650,5,1
2,20,A Time of Torment (Charlie Parker #14),48.35,5100.925,5,1
3,65,While You Were Mine,41.32,4359.260,5,1
4,60,The Red Tent,35.66,3762.130,5,1
5,48,Mrs. Houdini,30.25,3191.375,5,1
6,57,The Passion of Dolssa,28.32,2987.760,5,1
7,11,"1,000 Places to See Before You Die",26.08,2751.440,5,1
8,29,What Happened on Beale Street (Secrets of the ...,25.37,2676.535,5,1
9,34,The Silkworm (Cormoran Strike #2),23.05,2431.775,5,1


In [14]:
# SQL Query 2: DISTINCT + IN
# Show the different categories included in the selected categories

query2 = """
SELECT DISTINCT category_name
FROM categories
WHERE category_name IN ('Travel', 'Mystery', 'Historical Fiction')
ORDER BY category_name
"""

result2 = pd.read_sql_query(query2, conn)

print("SQL Query 2:")
print(query2)

print("\nQuery 2 Output:")
display(result2)

SQL Query 2:

SELECT DISTINCT category_name
FROM categories
WHERE category_name IN ('Travel', 'Mystery', 'Historical Fiction')
ORDER BY category_name


Query 2 Output:


,category_name
0,Historical Fiction
1,Mystery
2,Travel


In [15]:
# SQL Query 3: Books priced between £20 and £40

query3 = """
SELECT
    book_id,
    title,
    price_gbp,
    price_inr,
    rating,
    in_stock
FROM books
WHERE price_gbp BETWEEN 20 AND 40
ORDER BY price_gbp DESC;
"""

result3 = pd.read_sql_query(query3, conn)

print("SQL Query 3:")
print(query3)

print("\nQuery 3 Output:")
display(result3)

SQL Query 3:

SELECT
    book_id,
    title,
    price_gbp,
    price_inr,
    rating,
    in_stock
FROM books
WHERE price_gbp BETWEEN 20 AND 40
ORDER BY price_gbp DESC;


Query 3 Output:


,book_id,title,price_gbp,price_inr,rating,in_stock
0,52,A Paris Apartment,39.01,4115.555,4,1
1,10,Neither Here nor There: Travels in Europe,38.95,4109.225,3,1
2,33,In the Woods (Dublin Murder Squad #1),38.38,4049.090,2,1
3,55,The Invention of Wings,37.34,3939.370,1,1
4,5,Under the Tuscan Sun,37.33,3938.315,3,1
5,47,The House by the Lake,36.95,3898.225,1,1
6,4,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.170,2,1
7,60,The Red Tent,35.66,3762.130,5,1
8,24,Most Wanted,35.28,3722.040,3,1
9,66,The Secret Healer,34.56,3646.080,3,1


In [16]:
# SQL Query 4: Books from selected categories

query4 = """
SELECT
    book_id,
    title,
    price_gbp,
    rating,
    in_stock,
    category_id
FROM books
WHERE category_id IN (
    SELECT category_id
    FROM categories
    WHERE category_name IN ('Travel', 'Mystery')
)
ORDER BY rating DESC, price_gbp DESC;
"""

result4 = pd.read_sql_query(query4, conn)

print("SQL Query 4:")
print(query4)

print("\nQuery 4 Output:")
display(result4)

SQL Query 4:

SELECT
    book_id,
    title,
    price_gbp,
    rating,
    in_stock,
    category_id
FROM books
WHERE category_id IN (
    SELECT category_id
    FROM categories
    WHERE category_name IN ('Travel', 'Mystery')
)
ORDER BY rating DESC, price_gbp DESC;


Query 4 Output:


,book_id,title,price_gbp,rating,in_stock,category_id
0,30,The Bachelor Girl's Guide to Murder (Herringfo...,52.30,5,1,2
1,20,A Time of Torment (Charlie Parker #14),48.35,5,1,2
2,11,"1,000 Places to See Before You Die",26.08,5,1,1
3,29,What Happened on Beale Street (Secrets of the ...,25.37,5,1,2
4,34,The Silkworm (Cormoran Strike #2),23.05,5,1,2
5,40,The Girl You Lost,12.29,5,1,2
6,39,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,57.70,4,1,2
7,8,A Year in Provence (Provence #1),56.88,4,1,1
8,14,The Past Never Ends,56.50,4,1,2
9,23,Murder at the 42nd Street Library (Raymond Amb...,54.36,4,1,2


In [17]:
# SQL Query 5: JOIN books with categories

query5 = """
SELECT
    b.book_id,
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock,
    c.category_name
FROM books b
JOIN categories c
    ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.price_gbp DESC
LIMIT 10;
"""

result5 = pd.read_sql_query(query5, conn)

print("SQL Query 5:")
print(query5)

print("\nQuery 5 Output:")
display(result5)

SQL Query 5:

SELECT
    b.book_id,
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock,
    c.category_name
FROM books b
JOIN categories c
    ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.price_gbp DESC
LIMIT 10;


Query 5 Output:


,book_id,title,price_gbp,price_inr,rating,in_stock,category_name
0,46,A Flight of Arrows (The Pathfinders #2),55.53,5858.415,5,1,Historical Fiction
1,30,The Bachelor Girl's Guide to Murder (Herringfo...,52.30,5517.650,5,1,Mystery
2,20,A Time of Torment (Charlie Parker #14),48.35,5100.925,5,1,Mystery
3,65,While You Were Mine,41.32,4359.260,5,1,Historical Fiction
4,60,The Red Tent,35.66,3762.130,5,1,Historical Fiction
5,48,Mrs. Houdini,30.25,3191.375,5,1,Historical Fiction
6,57,The Passion of Dolssa,28.32,2987.760,5,1,Historical Fiction
7,11,"1,000 Places to See Before You Die",26.08,2751.440,5,1,Travel
8,29,What Happened on Beale Street (Secrets of the ...,25.37,2676.535,5,1,Mystery
9,34,The Silkworm (Cormoran Strike #2),23.05,2431.775,5,1,Mystery


In [18]:
# Read SQL Query 1 result into pandas
pandas_result1 = pd.read_sql(query1, conn)

print("Query 1 result using pd.read_sql():")
display(pandas_result1)

Query 1 result using pd.read_sql():


,book_id,title,price_gbp,price_inr,rating,in_stock
0,46,A Flight of Arrows (The Pathfinders #2),55.53,5858.415,5,1
1,30,The Bachelor Girl's Guide to Murder (Herringfo...,52.30,5517.650,5,1
2,20,A Time of Torment (Charlie Parker #14),48.35,5100.925,5,1
3,65,While You Were Mine,41.32,4359.260,5,1
4,60,The Red Tent,35.66,3762.130,5,1
5,48,Mrs. Houdini,30.25,3191.375,5,1
6,57,The Passion of Dolssa,28.32,2987.760,5,1
7,11,"1,000 Places to See Before You Die",26.08,2751.440,5,1
8,29,What Happened on Beale Street (Secrets of the ...,25.37,2676.535,5,1
9,34,The Silkworm (Cormoran Strike #2),23.05,2431.775,5,1


In [19]:
# Read SQL Query 5 result into pandas
pandas_result5 = pd.read_sql(query5, conn)

print("Query 5 result using pd.read_sql():")
display(pandas_result5)

Query 5 result using pd.read_sql():


,book_id,title,price_gbp,price_inr,rating,in_stock,category_name
0,46,A Flight of Arrows (The Pathfinders #2),55.53,5858.415,5,1,Historical Fiction
1,30,The Bachelor Girl's Guide to Murder (Herringfo...,52.30,5517.650,5,1,Mystery
2,20,A Time of Torment (Charlie Parker #14),48.35,5100.925,5,1,Mystery
3,65,While You Were Mine,41.32,4359.260,5,1,Historical Fiction
4,60,The Red Tent,35.66,3762.130,5,1,Historical Fiction
5,48,Mrs. Houdini,30.25,3191.375,5,1,Historical Fiction
6,57,The Passion of Dolssa,28.32,2987.760,5,1,Historical Fiction
7,11,"1,000 Places to See Before You Die",26.08,2751.440,5,1,Travel
8,29,What Happened on Beale Street (Secrets of the ...,25.37,2676.535,5,1,Mystery
9,34,The Silkworm (Cormoran Strike #2),23.05,2431.775,5,1,Mystery


In [20]:
# Reproduce the SQL JOIN using pandas

books_pandas = pd.read_sql("""
SELECT
    book_id,
    title,
    price_gbp,
    price_inr,
    rating,
    in_stock,
    category_id
FROM books
""", conn)

categories_pandas = pd.read_sql("""
SELECT
    category_id,
    category_name
FROM categories
""", conn)

# Merge books and categories using category_id
merged_result = pd.merge(
    books_pandas,
    categories_pandas,
    on="category_id",
    how="inner"
)

# Sort and limit to match SQL Query 5
merged_result = merged_result.sort_values(
    by=["rating", "price_gbp"],
    ascending=[False, False]
).head(10)

# Select same columns as SQL Query 5
merged_result = merged_result[
    [
        "book_id",
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category_name"
    ]
]

print("JOIN result reproduced using pandas pd.merge():")
display(merged_result)

JOIN result reproduced using pandas pd.merge():


,book_id,title,price_gbp,price_inr,rating,in_stock,category_name
45,46,A Flight of Arrows (The Pathfinders #2),55.53,5858.415,5,1,Historical Fiction
29,30,The Bachelor Girl's Guide to Murder (Herringfo...,52.30,5517.650,5,1,Mystery
19,20,A Time of Torment (Charlie Parker #14),48.35,5100.925,5,1,Mystery
64,65,While You Were Mine,41.32,4359.260,5,1,Historical Fiction
59,60,The Red Tent,35.66,3762.130,5,1,Historical Fiction
47,48,Mrs. Houdini,30.25,3191.375,5,1,Historical Fiction
56,57,The Passion of Dolssa,28.32,2987.760,5,1,Historical Fiction
10,11,"1,000 Places to See Before You Die",26.08,2751.440,5,1,Travel
28,29,What Happened on Beale Street (Secrets of the ...,25.37,2676.535,5,1,Mystery
33,34,The Silkworm (Cormoran Strike #2),23.05,2431.775,5,1,Mystery


In [21]:
# Compare SQL JOIN result with pandas merge result

sql_join_result = result5.reset_index(drop=True)
pandas_join_result = merged_result.reset_index(drop=True)

print("SQL JOIN result:")
display(sql_join_result)

print("Pandas merge() result:")
display(pandas_join_result)

print(
    "Do SQL JOIN and pandas merge() produce equivalent results?",
    sql_join_result.equals(pandas_join_result)
)

SQL JOIN result:


,book_id,title,price_gbp,price_inr,rating,in_stock,category_name
0,46,A Flight of Arrows (The Pathfinders #2),55.53,5858.415,5,1,Historical Fiction
1,30,The Bachelor Girl's Guide to Murder (Herringfo...,52.30,5517.650,5,1,Mystery
2,20,A Time of Torment (Charlie Parker #14),48.35,5100.925,5,1,Mystery
3,65,While You Were Mine,41.32,4359.260,5,1,Historical Fiction
4,60,The Red Tent,35.66,3762.130,5,1,Historical Fiction
5,48,Mrs. Houdini,30.25,3191.375,5,1,Historical Fiction
6,57,The Passion of Dolssa,28.32,2987.760,5,1,Historical Fiction
7,11,"1,000 Places to See Before You Die",26.08,2751.440,5,1,Travel
8,29,What Happened on Beale Street (Secrets of the ...,25.37,2676.535,5,1,Mystery
9,34,The Silkworm (Cormoran Strike #2),23.05,2431.775,5,1,Mystery


Pandas merge() result:


,book_id,title,price_gbp,price_inr,rating,in_stock,category_name
0,46,A Flight of Arrows (The Pathfinders #2),55.53,5858.415,5,1,Historical Fiction
1,30,The Bachelor Girl's Guide to Murder (Herringfo...,52.30,5517.650,5,1,Mystery
2,20,A Time of Torment (Charlie Parker #14),48.35,5100.925,5,1,Mystery
3,65,While You Were Mine,41.32,4359.260,5,1,Historical Fiction
4,60,The Red Tent,35.66,3762.130,5,1,Historical Fiction
5,48,Mrs. Houdini,30.25,3191.375,5,1,Historical Fiction
6,57,The Passion of Dolssa,28.32,2987.760,5,1,Historical Fiction
7,11,"1,000 Places to See Before You Die",26.08,2751.440,5,1,Travel
8,29,What Happened on Beale Street (Secrets of the ...,25.37,2676.535,5,1,Mystery
9,34,The Silkworm (Cormoran Strike #2),23.05,2431.775,5,1,Mystery


Do SQL JOIN and pandas merge() produce equivalent results? True
